# Inventory Risk Monitoring

## Objective

Monitor inventory health and identify inventory shortage risks using Days of Inventory (DOI), inventory coverage policies, and inventory alert rules.

## Data Sources

- inventory_snapshot_weekly.csv
- inventory_policy.csv

## Deliverables

- DOI (Days of Inventory)
- Inventory Status
- Reorder Risk
- Inventory Risk Alerts

In [54]:
import pandas as pd
import numpy as np

In [55]:
inventory_snapshot = pd.read_csv(
    "../data/processed/inventory_snapshot_weekly.csv"
)

inventory_policy = pd.read_csv(
    "../data/master/inventory_policy.csv"
)

In [56]:
inventory_risk = (
    inventory_snapshot.merge(
        inventory_policy[
            [
                "Product Card Id",
                "Safety_Stock_Days"
            ]
        ],
        on="Product Card Id",
        how="left"
    )
)

In [57]:
# Inventory remaining after weekly demand consumption

inventory_risk["Inventory_On_Hand"] = (
    inventory_risk["Ending_Inventory"]
)

In [58]:
# Number of days inventory can support average demand

inventory_risk["DOI"] = (
    inventory_risk["Inventory_On_Hand"]
    /
    inventory_risk["Average_Daily_Demand"]
)

## Inventory Status

Business Logic

Normal:
DOI >= Target Coverage Days

Warning:
Safety Stock Days <= DOI < Target Coverage Days

Critical:
DOI < Safety Stock Days

In [59]:
# Inventory health classification based on DOI

def classify_inventory_status(row):

    if row["DOI"] < row["Safety_Stock_Days"]:
        return "Critical"

    elif row["DOI"] < row["Target_Coverage_Days"]:
        return "Warning"

    else:
        return "Normal"


inventory_risk["Inventory_Status"] = (
    inventory_risk.apply(
        classify_inventory_status,
        axis=1
    )
)

In [60]:
inventory_risk.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Initial_Inventory,Target_Coverage_Days,Average_Daily_Demand,Review_Cycle,Beginning_Inventory,Ending_Inventory,Replenishment_Qty,Target_Inventory,Safety_Stock_Days,Inventory_On_Hand,DOI,Inventory_Status
0,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-04-30,2,48,21,0.457143,0,48.0,46.0,0,10,7,46.0,100.6250,Normal
1,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-07,3,48,21,0.457143,0,46.0,43.0,0,10,7,43.0,94.0625,Normal
2,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-14,1,48,21,0.457143,0,43.0,42.0,0,10,7,42.0,91.8750,Normal
3,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-21,3,48,21,0.457143,0,42.0,39.0,0,10,7,39.0,85.3125,Normal
4,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-28,6,48,21,0.457143,1,39.0,33.0,0,10,7,33.0,72.1875,Normal


In [61]:
# Indicates whether inventory has fallen below safety stock level

inventory_risk["Reorder_Risk"] = np.where(
    inventory_risk["DOI"]
    <
    inventory_risk["Safety_Stock_Days"],
    "Yes",
    "No"
)

In [62]:
inventory_risk["Inventory_Status"].value_counts()

Inventory_Status
Critical    3252
Warning     3164
Normal      1192
Name: count, dtype: int64

In [63]:
product_abc = pd.read_csv(r'../data/processed/product_abc_classification.csv')

In [64]:
inventory_risk = (
    inventory_risk.merge(
        product_abc[
            [
                "Product Card Id",
                "ABC_Class"
            ]
        ],
        on="Product Card Id",
        how="left"
    )
)

In [68]:
inventory_risk["DOI"].describe()

count    7608.000000
mean       18.930436
std        22.356811
min         0.000000
25%         2.622291
50%        12.938895
75%        23.571161
max       104.154589
Name: DOI, dtype: float64

In [70]:
print(inventory_risk.groupby(
    "ABC_Class"
)["DOI"].describe())

            count       mean        std        min        25%        50%  \
ABC_Class                                                                  
A           725.0  29.323575  11.631378  10.459924  21.813568  29.734660   
B          2268.0  16.396449  14.693348   0.000000   6.523786  15.267148   
C          4615.0  18.543016  26.009042   0.000000   0.000000   8.915094   

                 75%         max  
ABC_Class                         
A          37.274922   86.399787  
B          22.914830   89.456869  
C          18.657204  104.154589  


In [71]:
inventory_risk["Inventory_Status"].value_counts(normalize=True)

Inventory_Status
Critical    0.427445
Warning     0.415878
Normal      0.156677
Name: proportion, dtype: float64

Inventory KPI

In [72]:
# Percentage of target inventory currently available

inventory_risk["Inventory_Utilization"] = (
    inventory_risk["Inventory_On_Hand"]
    /
    inventory_risk["Target_Inventory"]
)

In [73]:
# Inventory alert flag for dashboard monitoring

inventory_risk["Inventory_Alert"] = np.where(
    inventory_risk["Inventory_Status"] == "Critical",
    1,
    0
)

In [74]:
inventory_risk.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Initial_Inventory,Target_Coverage_Days,Average_Daily_Demand,Review_Cycle,Beginning_Inventory,...,Replenishment_Qty,Target_Inventory,Safety_Stock_Days,Inventory_On_Hand,DOI,Inventory_Status,Reorder_Risk,ABC_Class,Inventory_Utilization,Inventory_Alert
0,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-04-30,2,48,21,0.457143,0,48.0,...,0,10,7,46.0,100.6250,Normal,No,C,4.6,0
1,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-07,3,48,21,0.457143,0,46.0,...,0,10,7,43.0,94.0625,Normal,No,C,4.3,0
2,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-14,1,48,21,0.457143,0,43.0,...,0,10,7,42.0,91.8750,Normal,No,C,4.2,0
3,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-21,3,48,21,0.457143,0,42.0,...,0,10,7,39.0,85.3125,Normal,No,C,3.9,0
4,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-28,6,48,21,0.457143,1,39.0,...,0,10,7,33.0,72.1875,Normal,No,C,3.3,0


In [ ]:
#inventory_risk.to_csv(
#    "../data/processed/inventory_risk_weekly.csv",
#    index=False)